# ACE-RAG Cloud Quickstart

Use this notebook on Kaggle or Google Colab. Enable GPU and internet before running.

Recommended first target: Kaggle T4 x2 or P100. If P100 throws `CUDA error: no kernel image is available for execution on the device`, switch to T4 x2 and restart. The current runner uses one model process, so dual T4 will not automatically double speed yet.

In [ ]:
!nvidia-smi

## Locate or Copy Project Folder

On Kaggle, upload `ace_rag_research` as a Dataset. If it is under `/kaggle/input`, this cell copies it to writable `/kaggle/working`.

In [ ]:
import glob
import os
import shutil
from pathlib import Path

def find_project():
    direct = [Path('/kaggle/working/ace_rag_research'), Path('/content/ace_rag_research')]
    for path in direct:
        if (path / 'ace_rag').exists():
            return path
    candidates = [Path(p) for p in glob.glob('/kaggle/input/**/ace_rag_research', recursive=True)]
    if candidates:
        dst = Path('/kaggle/working/ace_rag_research')
        if not dst.exists():
            shutil.copytree(candidates[0], dst)
        return dst
    raise FileNotFoundError('Could not find ace_rag_research. Upload the folder or zip first.')

PROJECT = find_project()
os.chdir(PROJECT)
print('PROJECT =', PROJECT)
!pwd
!ls

## Install Cloud Dependencies

In [ ]:
!python -m pip install -q -r requirements-cloud.txt
!python -m scripts.cloud_check

## Offline Smoke Test

In [ ]:
!python -m experiments.run_mvp --config configs/toy.yaml --out-dir cloud_results

## First Real HotpotQA Run

Start with `--limit 200`. Scale to 1000 only after this finishes cleanly.

In [ ]:
!python -m experiments.run_mvp \
  --config configs/hotpotqa.yaml \
  --limit 200 \
  --out-dir cloud_results

## Inspect and Save Results

In [ ]:
!ls -lh cloud_results
!python - <<'PY'
from pathlib import Path
for path in sorted(Path('cloud_results').glob('*metrics.csv')):
    print('\n---', path, '---')
    print(path.read_text())
PY
!zip -r ace_rag_cloud_results.zip cloud_results